# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **Can you modify the notebook to include the top 10 311 request by ward instead?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-05-20 18:48:44 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-05-20T18:48:44.895851")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `311 requests`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 2 datasets matching '311 requests'

1. **311 Data Archive**
   ID: `311-data`
   **NOTE: THIS DATA STOPPED UPDATING AS OF TUESDAY FEBRUARY 4, 2025** The City of Pittsburgh has transitioned to a new 311 system. We will be working on a new data connection to the
   - 311 Data (CSV) [DataStore] ID: `29462525-62a6-45bf-9b5e-ad2e1c06348d`
   - 311 Data User Guide (HTML) ID: `5601f50b-05c5-4e73-a1a7-bb5cbb54ae3c`
   - 311 Issue and Category Codebook (CSV) [DataStore] ID: `6a2c9de6-9cb8-4da6-bd78-0a913cc5790c`
   - 311 Data (old format, 2015-2022) (CSV) [DataStore] ID: `76fda9d0-69be-4dd5-8108
```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": '311 requests', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 5

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 937,565
Loaded: 5
Fields (25): case_number, status, case_owner, subject, subject_code, legacy_subject, created_date_et, created_date_utc, last_modified_date_et, last_modified_date_utc, closed_date_et, closed_date_utc, origin, street, street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy, unique_id

Sample (5 rows):

case_number status     case_owner     subject subject_code                   legacy_subject     created_date_et    created_date_utc last_modified_date_et last
```


In [ ]:
# Step 2: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 3: Load Data from Resource

**Resource ID:** `29462525-62a6-45bf-9b5e-ad2e1c06348d`
**Limit:** 5

**Result preview:**
```
Resource: 29462525-62a6-45bf-9b5e-ad2e1c06348d
Total records: 815,417
Loaded: 5
Fields (28): group_id, num_requests, parent_closed, status_name, status_code, dept, request_type_name, request_type_id, create_date_et, create_date_utc, last_action_et, last_action_utc, closed_date_et, closed_date_utc, origin, street, cross_street, street_id, cross_street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy

Sample (5 rows):

group_id  num_requests  parent_closed status_name  status_code                 dept request_type_name request_type_id  
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '29462525-62a6-45bf-9b5e-ad2e1c06348d', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 4: SQL Analysis Query

**SQL:**
```sql

SELECT 
  ward,
  subject AS request_type,
  COUNT(*) AS total_requests
FROM "5202679a-d243-402e-b82a-63189995a942"
WHERE ward IS NOT NULL AND ward != '' AND ward != 'NaN'
GROUP BY ward, subject
ORDER BY ward, total_requests DESC

```

**Result preview:**
```
SQL: 
SELECT 
  ward,
  subject AS request_type,
  COUNT(*) AS total_requests
FROM "5202679a-d243-402e-b82a-63189995a942"
WHERE ward IS NOT NULL AND ward != '' AND ward != 'NaN'
GROUP BY ward, subject
ORDER BY ward, total_requests DESC

Rows: 4158
Columns: ward, request_type, total_requests

ward                                       request_type  total_requests
   1                                           Potholes             804
   1                                    Illegal Parking             620
   1                                           Homeless             505
   1               
```


In [ ]:
# Step 4: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  ward,\n  subject AS request_type,\n  COUNT(*) AS total_requests\nFROM "5202679a-d243-402e-b82a-63189995a942"\nWHERE ward IS NOT NULL AND ward != \'\' AND ward != \'NaN\'\nGROUP BY ward, subject\nORDER BY ward, total_requests DESC\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 5: SQL Analysis Query

**SQL:**
```sql

SELECT ward, request_type, total_requests
FROM (
  SELECT 
    ward,
    subject AS request_type,
    COUNT(*) AS total_requests,
    ROW_NUMBER() OVER (PARTITION BY ward ORDER BY COUNT(*) DESC) AS rank
  FROM "5202679a-d243-402e-b82a-63189995a942"
  WHERE ward IS NOT NULL AND ward != '' AND ward != 'NaN'
  GROUP BY ward, subject
) ranked
WHERE rank <= 10
ORDER BY ward, total_requests DESC

```

**Result preview:**
```
SQL: 
SELECT ward, request_type, total_requests
FROM (
  SELECT 
    ward,
    subject AS request_type,
    COUNT(*) AS total_requests,
    ROW_NUMBER() OVER (PARTITION BY ward ORDER BY COUNT(*) DESC) AS rank
  FROM "5202679a-d243-402e-b82a-63189995a942"
  WHERE ward IS NOT NULL AND ward != '' AND ward != 'NaN'
  GROUP BY ward, subject
) ranked
WHERE rank <= 10
ORDER BY ward, total_requests DESC

Rows: 320
Columns: ward, request_type, total_requests

ward                             request_type  total_requests
   1                                 Potholes             804
   1                 
```


In [ ]:
# Step 5: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT ward, request_type, total_requests\nFROM (\n  SELECT \n    ward,\n    subject AS request_type,\n    COUNT(*) AS total_requests,\n    ROW_NUMBER() OVER (PARTITION BY ward ORDER BY COUNT(*) DESC) AS rank\n  FROM "5202679a-d243-402e-b82a-63189995a942"\n  WHERE ward IS NOT NULL AND ward != \'\' AND ward != \'NaN\'\n  GROUP BY ward, subject\n) ranked\nWHERE rank <= 10\nORDER BY ward, total_requests DESC\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here are the **Top 10 311 Requests by Ward** across Pittsburgh, based on the **Pittsburgh 311 Data** dataset on the WPRDC portal (937,565 total records):

---

## 🗺️ Top 10 311 Requests by Ward

| Ward | #1 Request | #2 | #3 | #4 | #5 | #6 | #7 | #8 | #9 | #10 |
|------|-----------|----|----|----|----|----|----|----|----|-----|
| **1** | Potholes (804) | Illegal Parking (620) | Homeless (505) | Weeds or Debris (459) | Litter on Public Property (452) | Building Maintenance (404) | Police - Submit a Tip (368) | Street Light Concerns (316) | Repair/Replace a Sign (307) | Fire Safety Concerns (282) |
| **2** | Illegal Parking (2,398) | Potholes (1,089) | Litter on Public Property (782) | Homeless (650) | Street Light Concerns (624) | Blocked Sidewalks/Trails (576) | Repair/Replace a Sign (441) | Police Concerns (432) | Traffic Signal Repair (417) | Fire Safety Concerns (413) |
| **10** | Potholes (4,062) | Missed Pick Up (4,003) | Weeds or Debris (2,303) | Abandoned Vehicle (1,495) | Snow/Ice Removal (1,278) | Building Maintenance (1,037) | City Tree Maintenance (1,003) | Street Light Concerns (886) | Refuse/Recycling Violations (871) | Paving Request (865) |
| **11** | Potholes (3,090) | Missed Pick Up (2,522) | Weeds or Debris (2,140) | Abandoned Vehicle (1,336) | City Tree Maintenance (935) | Building Maintenance (870) | Refuse/Recycling Violations (862) | Street Light Concerns (821) | Street Cleaning/Sweeping (740) | Illegal Parking (700) |
| **12** | Weeds or Debris (2,731) | Potholes (2,509) | City Owned Property Maint. (2,243) | Missed Pick Up (1,569) | Building Maintenance (998) | Snow/Ice Removal (953) | Abandoned Vehicle (848) | Litter on Public Property (640) | Overgrowth (627) | Street Light Concerns (572) |
| **13** | Weeds or Debris (3,958) | Potholes (2,404) | City Owned Property Maint. (1,965) | Building Maintenance (1,661) | Missed Pick Up (1,491) | Snow/Ice Removal (1,158) | Abandoned Vehicle (904) | Litter (770) | Vacant Buildings (674) | Narcotics/Vice/Gang Issues (559) |
| **14** | Potholes (7,807) | Missed Pick Up (4,836) | Weeds or Debris (3,178) | City Tree Maintenance (2,813) | Snow/Ice Removal (2,497) | Refuse/Recycling (2,210) | Street Light Concerns (1,963) | Repair/Replace a Sign (1,828) | Street Cleaning (1,814) | Parks & Playground Maint. (1,663) |
| **16** | Weeds or Debris (8,304) | Refuse/Recycling Violations (6,335) | Missed Pick Up (2,511) | Potholes (2,202) | Snow/Ice Removal (2,012) | Building Maintenance (1,327) | Abandoned Vehicle (1,103) | Litter (773) | Work Without Permits (770) | Illegal Parking (694) |
| **17** | Refuse/Recycling Violations (7,159) | Weeds or Debris (6,598) | Missed Pick Up (1,596) | Potholes (1,441) | Work Without Permits (927) | Building Maintenance (863) | Illegal Parking (768) | Snow/Ice Removal (726) | Litter (689) | Repair/Replace a Sign (662) |
| **19** | Missed Pick Up (6,537) | Weeds or Debris (6,394) | Potholes (6,087) | Snow/Ice Removal (5,125) | Abandoned Vehicle (2,863) | Illegal Parking (2,329) | Refuse/Recycling (2,243) | Building Maintenance (2,242) | Street Light Concerns (1,572) | Repair/Replace a Sign (1,505) |

---

### 🔑 Key Observations:
- **Potholes, Weeds or Debris, and Missed Pick Ups** are the most universally common complaints across nearly all wards.
- **Ward 14** has the single highest pothole count (7,807), and **Ward 16** leads in Weeds or Debris (8,304).
- **Wards 1 & 2** stand out with high **Homeless** and **Police** related requests, typical of more urban/downtown wards.
- **Refuse/Recycling Violations** dominate Wards 16 & 17, suggesting enforcement challenges in those areas.

> 📌 **Source:** *Pittsburgh 311 Data* — WPRDC (data.wprdc.org) | Note: Data covers the new 311 system (Feb 4, 2025 onward) combined with historical records migrated into the new system. Some records may have missing ward data and were excluded.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-05-20

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-05-20 18:48:44
- **Query**: Can you modify the notebook to include the top 10 311 request by ward instead?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
